In [ ]:
import base64
from dotenv import load_dotenv
import os

from anthropic import Anthropic
from anthropic.types import Message

In [7]:
load_dotenv(override=True)

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")

client = Anthropic(api_key=anthropic_api_key)

model = "claude-sonnet-4-5-20250929"

In [8]:
def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)

In [9]:
def chat(
    messages,
    system=None,
    temperature=1.0,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=1024    # max tokens for thinking when claude generates the response (1024 is min)
    ):
    params = {
        "model": model,
        "max_tokens": 4000,     # must be higher than 'thinking_budget'
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "adaptive",
            "budget_tokens": thinking_budget
        }

    if tools:
        params["tools"] = tools
    
    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message

def text_from_message(message):
    return "\n".join(
        [block for block in message.content if block.type == "text"]
    )
    

In [12]:
with open("../../data/documents/earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8") 

messages = []

add_user_message(messages, [
    {
        "type": "document",
        "source": {
            "type": "base64",
            "media_type": "application/pdf",
            "data": file_bytes,
        },
        "title": "earth.pdf",
        "citations": {"enabled": True}
    },
    {
        "type": "text",
        "text": "How were Earth's atmosphere and oceans formed?"
    }
])

response = chat(messages)

In [13]:
citations = [
    citation
    for block in response.content
    if block.citations
    for citation in block.citations
]

In [14]:
for citation in citations:
    print(citation.cited_text)
    print(citation.document_title)
    print(citation.start_page_number)

[42]
Earth's atmosphere and oceans were formed by volcanic activity and outgassing.

earth.pdf
4
[43] Water vapor from
these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets,
and comets.

earth.pdf
4


In [11]:
with open("../../data/documents/earth.pdf", "rb") as f:
    file_bytes = base64.standard_b64encode(f.read()).decode("utf-8")

messages = []

add_user_message(
    messages,
    [
        {
            "type": "document",
            "source": {
                "type": "text",
                "media_type": "text/plain",
                "data": article_text,
            },
            "title": "Earth Article",
            "citations": {"enabled": True},
        },
        {
            "type": "text",
            "text": "How were Earth's atmosphere and oceans were formed?",
        },
    ],
)

chat(messages)

Message(id='msg_011CfruKEEbQown7o7i59YzZ', container=None, content=[TextBlock(citations=[CitationCharLocation(cited_text="[42]\nAfter formation\nEarth's atmosphere and oceans were formed by volcanic activity and outgassing.\n", document_index=0, document_title='Earth Article', end_char_index=9977, file_id=None, start_char_index=9877, type='char_location')], text="Earth's atmosphere and oceans were formed by volcanic activity and outgassing.", type='text'), TextBlock(citations=None, text=' ', type='text'), TextBlock(citations=[CitationCharLocation(cited_text='[43] Water vapor from\nthese sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets,\nand comets.\n', document_index=0, document_title='Earth Article', end_char_index=10109, file_id=None, start_char_index=9977, type='char_location')], text='Water vapor from these sources condensed into the oceans, augmented by water and ice from asteroids, protoplanets, and comets.', type='text')], model='claude-